In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle

In [2]:
Path("../data/graph").mkdir(parents=True, exist_ok=True)

In [3]:
train_df = pd.read_csv("../data/processed/train_1_50.csv", low_memory=False)
test_df = pd.read_csv("../data/processed/test_temporal.csv", low_memory=False)

print("Train:", train_df.shape)
print(train_df["label_final"].value_counts())

print("Test:", test_df.shape)
print(test_df["label_final"].value_counts())

Train: (38454, 25)
label_final
0    37700
1      754
Name: count, dtype: int64
Test: (298160, 25)
label_final
0    298096
1        64
Name: count, dtype: int64


# Memastikan Time Stamp urut

In [4]:
train_df = train_df.sort_values("timestamp").reset_index(drop=True)
test_df = test_df.sort_values("timestamp").reset_index(drop=True)

# menentukan node : wallet address

In [5]:
all_wallets = pd.concat([
    train_df["from_address"],
    train_df["to_address"],
    test_df["from_address"],
    test_df["to_address"]
]).dropna().astype(str).str.lower().unique()

wallet_to_id = {
    wallet: idx for idx, wallet in enumerate(all_wallets)
}

id_to_wallet = {
    idx: wallet for wallet, idx in wallet_to_id.items()
}

num_nodes = len(wallet_to_id)

print("Total nodes:", num_nodes)

Total nodes: 108582


# Mapping wallet ke node id

In [6]:
def map_wallets(df):
    df = df.copy()
    df["source"] = df["from_address"].astype(str).str.lower().map(wallet_to_id)
    df["target"] = df["to_address"].astype(str).str.lower().map(wallet_to_id)
    return df

train_df = map_wallets(train_df)
test_df = map_wallets(test_df)

print(train_df[["from_address", "to_address", "source", "target"]].head())

                                 from_address  \
0  0x4D93C788B6E9771F1ee2f30242cD3892b631d8ed   
1  0xD387A6E4e84a6C86bd90C158C6028A58CC8Ac459   
2  0x283Af0B28c62C092C9727F1Ee09c02CA627EB7F5   
3  0x4D93C788B6E9771F1ee2f30242cD3892b631d8ed   
4  0x283Af0B28c62C092C9727F1Ee09c02CA627EB7F5   

                                   to_address  source  target  
0  0xC8803d21A704BFeBdBC394bD16501a4b36aD3A2D       0     176  
1  0xEBE326d8De3413F8132518dcFd45e6cBFf7E5c27       1   14748  
2  0x08a7aD00DAc20aAAeB0612Ef3b96b737fE742d4F       2   14749  
3  0xa882F86CE9d1a8F8304666217d048247bB787770       0   14750  
4  0x2b39694f5014A06773d8BC491715c2FE81d11668       2   14751  


# Feature Engineering

In [10]:
# Pastikan timestamp numerik
train_df["timestamp"] = pd.to_numeric(train_df["timestamp"], errors="coerce")
test_df["timestamp"] = pd.to_numeric(test_df["timestamp"], errors="coerce")

train_df["mint_timestamp"] = pd.to_numeric(train_df["mint_timestamp"], errors="coerce")
test_df["mint_timestamp"] = pd.to_numeric(test_df["mint_timestamp"], errors="coerce")

In [11]:
# 1. holding_time_hours
train_df["holding_time_hours"] = (
    train_df["timestamp"] - train_df["mint_timestamp"]
) / 3600

test_df["holding_time_hours"] = (
    test_df["timestamp"] - test_df["mint_timestamp"]
) / 3600

train_df["holding_time_hours"] = train_df["holding_time_hours"].clip(lower=0)
test_df["holding_time_hours"] = test_df["holding_time_hours"].clip(lower=0)

In [12]:
# 2. pair_frequency
pair_cols = ["nft_address", "token_id", "from_address", "to_address"]

train_df["pair_frequency"] = (
    train_df.groupby(pair_cols)["transaction_hash"]
    .transform("count")
)

test_df["pair_frequency"] = (
    test_df.groupby(pair_cols)["transaction_hash"]
    .transform("count")
)

In [13]:
# 3. time_since_last_hours
def add_time_since_last(df):
    df = df.sort_values(["nft_address", "token_id", "timestamp"]).copy()
    
    df["prev_timestamp"] = (
        df.groupby(["nft_address", "token_id"])["timestamp"]
        .shift(1)
    )
    
    df["time_since_last_hours"] = (
        df["timestamp"] - df["prev_timestamp"]
    ) / 3600
    
    df = df.drop(columns=["prev_timestamp"])
    
    return df

train_df = add_time_since_last(train_df)
test_df = add_time_since_last(test_df)

In [14]:
# 4. symmetry_ratio_from
epsilon = 1e-6

train_df["symmetry_ratio_from"] = (
    train_df["transfers_in_from"] /
    (train_df["transfers_out_from"] + epsilon)
)

test_df["symmetry_ratio_from"] = (
    test_df["transfers_in_from"] /
    (test_df["transfers_out_from"] + epsilon)
)

train_df["symmetry_ratio_from"] = train_df["symmetry_ratio_from"].clip(upper=100)
test_df["symmetry_ratio_from"] = test_df["symmetry_ratio_from"].clip(upper=100)

In [15]:
feature_cols = [
    "transaction_value",
    "holding_time_hours",
    "pair_frequency",
    "time_since_last_hours",
    "symmetry_ratio_from",
    "transfers_out_from",
    "transfers_in_from",
    "transfers_out_to",
    "transfers_in_to",
    "num_transitions"
]

In [16]:
missing_cols = [col for col in feature_cols if col not in train_df.columns]

print("Missing columns:", missing_cols)

Missing columns: []


# normalization

In [17]:
for col in feature_cols:
    train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
    test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

train_df[feature_cols] = train_df[feature_cols].fillna(0)
test_df[feature_cols] = test_df[feature_cols].fillna(0)

In [18]:
feature_mean = train_df[feature_cols].mean()
feature_std = train_df[feature_cols].std().replace(0, 1)

train_df[feature_cols] = (
    train_df[feature_cols] - feature_mean
) / feature_std

test_df[feature_cols] = (
    test_df[feature_cols] - feature_mean
) / feature_std

In [19]:
def build_graph_dict(df, feature_cols):
    graph = {
        "edge_index": df[["source", "target"]].values.T.astype(np.int64),
        "edge_features": df[feature_cols].values.astype(np.float32),
        "edge_labels": df["label_final"].values.astype(np.int64),
        "timestamps": df["timestamp"].values.astype(np.float32),
        "transaction_hash": df["transaction_hash"].values,
        "source": df["source"].values.astype(np.int64),
        "target": df["target"].values.astype(np.int64),
    }
    return graph

train_graph = build_graph_dict(train_df, feature_cols)
test_graph = build_graph_dict(test_df, feature_cols)

In [20]:
print("Train edge_index:", train_graph["edge_index"].shape)
print("Train edge_features:", train_graph["edge_features"].shape)
print("Train labels:", train_graph["edge_labels"].shape)

print("Test edge_index:", test_graph["edge_index"].shape)
print("Test edge_features:", test_graph["edge_features"].shape)
print("Test labels:", test_graph["edge_labels"].shape)

Train edge_index: (2, 38454)
Train edge_features: (38454, 10)
Train labels: (38454,)
Test edge_index: (2, 298160)
Test edge_features: (298160, 10)
Test labels: (298160,)


In [21]:
with open("../data/graph/train_graph.pkl", "wb") as f:
    pickle.dump(train_graph, f)

with open("../data/graph/test_graph.pkl", "wb") as f:
    pickle.dump(test_graph, f)

with open("../data/graph/wallet_mapping.pkl", "wb") as f:
    pickle.dump({
        "wallet_to_id": wallet_to_id,
        "id_to_wallet": id_to_wallet,
        "num_nodes": num_nodes
    }, f)

with open("../data/graph/feature_scaler.pkl", "wb") as f:
    pickle.dump({
        "feature_cols": feature_cols,
        "mean": feature_mean,
        "std": feature_std
    }, f)

print("Graph dataset saved.")

Graph dataset saved.


In [22]:
summary = {
    "num_nodes": num_nodes,
    "train_edges": train_graph["edge_index"].shape[1],
    "test_edges": test_graph["edge_index"].shape[1],
    "num_edge_features": len(feature_cols),
    "train_positive": int(train_graph["edge_labels"].sum()),
    "train_negative": int((train_graph["edge_labels"] == 0).sum()),
    "test_positive": int(test_graph["edge_labels"].sum()),
    "test_negative": int((test_graph["edge_labels"] == 0).sum()),
}

summary

{'num_nodes': 108582,
 'train_edges': 38454,
 'test_edges': 298160,
 'num_edge_features': 10,
 'train_positive': 754,
 'train_negative': 37700,
 'test_positive': 64,
 'test_negative': 298096}